# 1. TextLoader

The simplest loader in LangChain. If you understand this one, every other loader is just "the same
idea, but for a fancier file format."

---

## 1. Simple Definition

> **Kid version:** You have a notebook page with words on it. `TextLoader` is a helper that **reads
> the whole page out loud** and hands you the words in a tidy box (a `Document`). That's it.

**Professional definition:** `TextLoader` reads a **plain-text file** (`.txt`, `.md`, `.py`, any
UTF-ish text) and returns a list containing **one `Document`** whose `page_content` is the file's
full text.

```python
from langchain_community.document_loaders import TextLoader

docs = TextLoader("notes.txt").load()
print(docs[0].page_content)          # the entire file's text
print(docs[0].metadata)              # {'source': 'notes.txt'}
```

---

## 2. Why Does It Exist?

**The problem:** Even reading a plain file "the LangChain way" needs to produce a `Document` with the
right metadata, handle file encodings, and plug into the same pipeline as every other loader.

### Before LangChain

```python
text = open("notes.txt", encoding="utf-8").read()
# Now what? It's just a string. No metadata, no standard shape,
# doesn't fit into splitters/vector stores without extra glue.
```

### After LangChain

```python
docs = TextLoader("notes.txt", encoding="utf-8").load()
# → [Document(page_content=<file text>, metadata={"source": "notes.txt"})]
# Standard shape, carries its source, ready for Split → Embed → Store.
```

You get the standard `Document` shape and `source` metadata for free, and the same `.load()`
interface as every other loader.

---

## 3. Real-Life Analogy

A **photocopier** 🖨️. You put one page in, and it gives you back an exact copy — plus a little stamp
saying which original it came from (the `source` metadata). It doesn't reorganize or interpret; it
just faithfully reproduces the text.

---

## 4. Where It Fits in LangChain Architecture

```
BaseLoader
    │
    ▼
TextLoader          ← reads plain text → 1 Document
```

- **`BaseLoader` → `TextLoader`:** the parent provides `.load()`/`.lazy_load()`; `TextLoader` just
  implements "open the file, read it, wrap it in one `Document`."
- Output: usually a **single** `Document` (the whole file), unlike CSV/PDF which produce many.

---

## 5. Internal Working

```
  "notes.txt"
        │
        ▼
  OPEN the file  (with the given encoding)
        │
        ▼
  READ all text  → "Hello world..."
        │
        ▼
  WRAP into ONE Document
     page_content = "Hello world..."
     metadata     = {"source": "notes.txt"}
        │
        ▼
  return [ Document ]
```

---

## 6. Attributes (constructor arguments)

### `file_path`

**Definition:** The path to the text file to read.

**Why it exists:** It's the source — the loader needs to know *what* to read.

**When developers use it:** Always (first argument).

**Real-life use case:** Telling the photocopier which page to copy.

In [1]:
from pprint import pprint

def pretty_print_doc(doc):
    print("=" * 80)
    print("📄 CONTENT")
    print("-" * 80)
    print(doc.page_content)

    print("\n🏷️ METADATA")
    print("-" * 80)
    pprint(doc.metadata)

    print("=" * 80)
    print()

In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(file_path = r"knowledge-source\transformers.txt")
loader

In [3]:
docs = loader.load()
print("length of docs: ", len(docs))
pretty_print_doc(docs[0])

length of docs:  1
📄 CONTENT
--------------------------------------------------------------------------------
# Transformer Model in Large Language Models (LLMs)

This note explains the Transformer model as used in modern Large Language Models (LLMs). It uses simple language and clear structure. You’ll get the key ideas, components, how training and inference work, strengths and limits, and common improvements.

---

## 1. Big picture - Why Transformers matter
- Transformers are the core architecture behind most modern LLMs (GPT, BERT, PaLM, LLaMA).
- They replaced recurrent and convolutional models because they handle long-range context efficiently and are highly parallelizable on GPUs/TPUs.
- The main innovation: self-attention, which lets the model weigh relationships among all tokens in a sequence.

---

## 2. Core building blocks
A Transformer layer consists of a few repeating parts:

1. Multi-Head Self-Attention
   - Computes attention scores between every pair of tokens.
   - Pr

### encoding

**Definition:** The character encoding used to decode the file (e.g. `"utf-8"`).

**Why it exists:** Text files can be saved in different encodings. The wrong one causes garbled
characters (mojibake) or a `UnicodeDecodeError` — very common on Windows files.

**When developers use it:** Whenever a file isn't the default encoding, or you hit decode errors.

**Real-life use case:** Choosing the right "alphabet" to read a foreign-language page correctly.


In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(file_path = r"knowledge-source\transformers.txt",
                    encoding="cp1252")
docs = loader.load()
pretty_print_doc(docs[0])

📄 CONTENT
--------------------------------------------------------------------------------
# Transformer Model in Large Language Models (LLMs)

This note explains the Transformer model as used in modern Large Language Models (LLMs). It uses simple language and clear structure. You’ll get the key ideas, components, how training and inference work, strengths and limits, and common improvements.

---

## 1. Big picture - Why Transformers matter
- Transformers are the core architecture behind most modern LLMs (GPT, BERT, PaLM, LLaMA).
- They replaced recurrent and convolutional models because they handle long-range context efficiently and are highly parallelizable on GPUs/TPUs.
- The main innovation: self-attention, which lets the model weigh relationships among all tokens in a sequence.

---

## 2. Core building blocks
A Transformer layer consists of a few repeating parts:

1. Multi-Head Self-Attention
   - Computes attention scores between every pair of tokens.
   - Produces a weighted s

### autodetect_encoding

**Definition:** If `True`, the loader tries to **guess** the file's encoding automatically.

**Why it exists:** When you load many files and don't know each one's encoding, autodetection avoids
crashes.

**When developers use it:** Bulk-loading mixed files (e.g. via `DirectoryLoader`).

**Real-life use case:** A smart scanner that figures out the language of a page by itself.

In [5]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(file_path = r"knowledge-source\transformers.txt",
                   autodetect_encoding=True)
docs = loader.load()
pretty_print_doc(docs[0])

📄 CONTENT
--------------------------------------------------------------------------------
# Transformer Model in Large Language Models (LLMs)

This note explains the Transformer model as used in modern Large Language Models (LLMs). It uses simple language and clear structure. You’ll get the key ideas, components, how training and inference work, strengths and limits, and common improvements.

---

## 1. Big picture - Why Transformers matter
- Transformers are the core architecture behind most modern LLMs (GPT, BERT, PaLM, LLaMA).
- They replaced recurrent and convolutional models because they handle long-range context efficiently and are highly parallelizable on GPUs/TPUs.
- The main innovation: self-attention, which lets the model weigh relationships among all tokens in a sequence.

---

## 2. Core building blocks
A Transformer layer consists of a few repeating parts:

1. Multi-Head Self-Attention
   - Computes attention scores between every pair of tokens.
   - Produces a weighted s